Thunderbird 원본(약 2.1억 줄)에서 v7 규격대로 Alert-Normal 짝을 뽑는 스트리밍 파서.

핵심 원칙
- alert(첫 칸이 '-'가 아닌 줄)만 수집하되, 연속 burst는 하나의 incident로 묶는다 (v7 §2).
- 각 incident alert target에 같은 node의 Normal target 1개를 매칭한다
  (v7 §4: same node 필수 + 시간/토큰길이 통제).

출력
- pairs.csv : 짝 목록 + 매칭 품질

이 스크립트는 "짝 추출"까지만 한다.
21-line 창 만들기(v7문서에 명시), 파일럿 임계값(2번), 시뮬 검정력(1번)은
여기서 나온 pairs.csv로 다음 단계에서 이어서 한다.

In [ ]:
import csv, re
from collections import deque, Counter
from pathlib import Path
from transformers import AutoTokenizer

Raw_path = Path("/Users/mac/Downloads/Thunderbird.log")
Out_path = Path("pairs_out/pairs.csv")
MODEL = "Qwen/Qwen2.5-3B-Instruct"

Burts_sec = 3600
match_time_sec = 3600
per_tempalte = 100
node_buf = 400
seed = 42

Out_path.parent.mkdir(parents=True, exist_ok = True)
tok = AutoTokenizer.from_pretrained(MODEL)

In [ ]:
def n_tok(text): # 토큰 개수 세기. autotikenizer사용. input_ids는 내부적으로 저장된 형태(토크나이저 고유 번호)
    return len(tok(text, add_sepecial_tokens=False)["input_ids"])
# add_sepecial_tokens 순수한 토큰개수만 세기.

_num = re.compile(r"0*[0-9a-fA-F]+|-?\d+") #템플릿 모양

def parse(raw):
    p = raw.split()
    if len(p) < 5: #최소 5조각은 있어야 함. [ㅈ]
        return None
    try:
        ts = int(p[1])
    except ValueError:
        return None
    content = raw.split(None, 8)[-1].rstrip("\n") if len(p) > 8 else raw.rstrip("\n")
    return p[0], ts, p[3], content

def template(content): # 템플릿 추출
    return _num.sub("<*>", content)    

In [ ]:
node_recent = {}
taken = Counter()
active = None
rows = []


def match_normal(a_idx, a_ts, a_node, a_content):
    a_tokens = n_tok(a_content)
    cands = [r for r in node_recent.get(a_node, [])
        if r[2] == "-" and r[0] != a_idx and abs(r[1] - a_ts) <= match_time_sec]
    if not cands:
        return None, a_tokens, None
    
    cands.sort(key=lambda r: (abs(len(r[3]) - len(a_content)), abs(r[1] - a_ts)))
    best, best_key = None, None
    for r in cands[:20]:
        key = (abs(n_tok(r[3]) - a_tokens), abs(r[1] - a_ts))
        if best_key is None or key < best_key:
            best_key, best = key, r
    return best, a_tokens, n_tok(best[3])

In [ ]:
with Raw_path.open(encoding="utf-8", errors="replace") as f:
    for idx, raw in enumerate(f):
        rec = parse(raw)
        if rec is None:
            continue
        label, ts, node, content = rec
        
        buf = node_recent.setdefault(node, deque(maxlen=node_buf))
        buf.append((idx,ts,label, content))
        
        if label == "-":
            continue
        
        sig = template(content)
        
        if active and active[0] == node and active[1] == sig and ts-active[2] <= Burts_sec:
            active = (node, sig, ts)
            continue
        active = (node, sig, ts)
        
        if taken[sig] >= per_tempalte:
            continue
        taken[sig] += 1
        
        normal, a_tokens, n_tokens = match_normal(idx, ts, node, content)
        pid = f"P{len(rows):05d}"
        if normal is None:
            rows.append([pid, "no_normal", sig[:80], idx, ts, node, a_tokens, 
                         "", "", "", "", content[:200]])
        else:
            rows.append([pid, "no_normal", sig[:80], idx, ts, node,
                         a_tokens, normal[0], normal[1], n_tokens,
                         abs(normal[1] - ts), content[:200]])

In [ ]:
header = ["pair_id", "status", "template", "alert_idx", "alert_ts", "node",
          "alert_tokens", "normal_idx", "normal_ts", "normal_tokens",
          "dt_sec", "alert_content"]
with Out_path.open("w", newline="", encoding="utf-8-sig") as fp:
    w = csv.writer(fp)
    w.writerow(header)
    w.writerows(rows)
 
ok = sum(1 for r in rows if r[1] == "ok")
print(f"짝 성공: {ok}   |   normal 없음: {len(rows) - ok}   |   종류: {len(taken)}")
print(f"저장: {Out_path}")